# translation of Alpaca dataset

In this notebook, we will machine translate RU version of cleaned Alpaca: https://huggingface.co/datasets/d0rj/alpaca-cleaned-ru

To do that, we will compare 3 models. Also we will pwerform translation assessment using fasttext.

## Imports

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate
!pip install fasttext -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from tqdm import tqdm
import json
import pandas as pd
from tqdm import tqdm
import re

## Dataset

In [ ]:
dataset = load_dataset("d0rj/alpaca-cleaned-ru", split="train")

# smal batch to compare models
test_samples = dataset.select(range(20))

print(len(dataset))

In [ ]:
print(type(test_samples[0]))
print(test_samples[0])
print(test_samples[0].keys())

## Models

In [ ]:
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    return tokenizer, model, device

In [ ]:
def translate_batch(texts, tokenizer, model, device, model_type):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)

    if model_type == "nllb":
        forced_bos = tokenizer.convert_tokens_to_ids("bak_Cyrl")

        generated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_length=256
        )

    elif model_type == "m2m":
        tokenizer.src_lang = "ru"
        forced_bos = tokenizer.get_lang_id("ba")

        generated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_length=256
        )

    elif model_type == "mbart":
        tokenizer.src_lang = "ru_RU"
        forced_bos = tokenizer.convert_tokens_to_ids("ru_RU")  # fallback (MBART bad for ba)

        generated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_length=256
        )

    else:
        raise ValueError("unknown model")

    return tokenizer.batch_decode(generated, skip_special_tokens=True)

## Test

In [ ]:
# MODELS = {
#     "nllb": "facebook/nllb-200-distilled-600M",
#     "m2m": "facebook/m2m100_418M",
#     "mbart": "facebook/mbart-large-50-many-to-many-mmt"
# }

In [ ]:
# results = {}

# for name, model_name in MODELS.items():
#     print(f"\n=== TESTING {name} ===")

#     tokenizer, model, device = load_model(model_name)

#     sample_inputs = [x["instruction"] for x in test_samples]
#     translations = translate_batch(sample_inputs, tokenizer, model, device, name)


#     results[name] = translations

#     for i in range(3):
#         print("\nRU:", sample_inputs[i])
#         print("BA:", translations[i])

In [ ]:
# ru_texts = [x["instruction"] for x in test_samples]

# table = pd.DataFrame({
#     "ru": ru_texts
# })

# for name, translations in results.items():
#     table[f"{name}_ba"] = translations



In [ ]:
# import pandas as pd

# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.max_colwidth", None)

In [ ]:
# table

### Results

Results were evaluated with 2 native speakers

- **facebook/nllb-200-distilled-600M**: almost ideal translation
- **facebook/m2m100_418M**: hallucinized?
- **facebook/mbart-large-50-many-to-many-mmt**: do not know bashkir

## Translate!

Use best model, filter, check using back translation

### Config

In [ ]:
TARGET_LANG = "bak_Cyrl"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 256

BEST_MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer_best, model_best, device_best = load_model(BEST_MODEL_NAME)

TOTAL_PARTS = 10
PART_ID = 9


OUTPUT_FILE = f"alpaca_ru_ba_part_{PART_ID}.jsonl"
CACHE_FILE = "translation_cache.json"


size = len(dataset)
part_size = size // TOTAL_PARTS

start = PART_ID * part_size
end = (PART_ID + 1) * part_size if PART_ID < TOTAL_PARTS - 1 else size

dataset_part = dataset.select(range(start, end))

print(f"Translating part {PART_ID}: {start} — {end}")

### Ensure safety via cache

In [ ]:
def load_cache():
    try:
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except:
        return {}

def save_cache(cache):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

### Translation functions

In [ ]:
def extract_fields(item):
    return item["instruction"], item["input"], item["output"]


def translate_item(item, tokenizer, model, device, cache):
    instr, inp, out = item["instruction"], item["input"], item["output"]

    def tr(text):
        if not text:
            return ""
        if text in cache:
            return cache[text]

        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

        forced_bos = tokenizer.convert_tokens_to_ids("bak_Cyrl")
        # print("forced_bos_token_id =", forced_bos)

        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_new_tokens=256
            )

        res = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        cache[text] = res
        return res

    return {
        "instruction": tr(instr),

        "input": tr(inp),

        "output": tr(out),
    }

# save format
def build_text(item):
    parts = []

    if item.get("instruction"):
        parts.append("instruction: " + item["instruction"])

    if item.get("input"):
        parts.append("input: " + item["input"])

    if item.get("output"):
        parts.append("output: " + item["output"])

    return "\n".join(parts)

### Run

In [ ]:
def run(dataset, tokenizer, model, device, log_every=10):
    cache = load_cache()
    results = []

    for i in tqdm(range(len(dataset))):
        item = dataset[i]

        instr, inp, out = item["instruction"], item["input"], item["output"]

        if instr in cache and inp in cache and out in cache:
            translated = {
                "instruction": cache.get(instr, ""),
                "input": cache.get(inp, ""),
                "output": cache.get(out, "")
            }
        else:
            translated = translate_item(item, tokenizer, model, device, cache)

        results.append(translated)

        if i % 50 == 0 and i > 0:
            save_cache(cache)

    save_cache(cache)
    return results

In [ ]:
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
translated = run(dataset_part, tokenizer_best, model_best, device_best)

save_jsonl(translated, OUTPUT_FILE)